In [ ]:
from IPython.core.display import Markdown
from IPython.core.display_functions import display
import pandas as pd
from pathlib import Path

""" métodos de verificación y carga de archivos """
def verificar_archivo(str) -> Path:
    # Archivo de encuesta de inversores
    ruta = Path("./csv/") / str
    
    # Verificar exitencia
    if ruta.exists():
            return ruta
            
    # Devolver error Archivo no encontrado
    raise FileNotFoundError(
        "No se encontró el directorio con los archivos del Proyecto C."
    )

def cargar_datos(archivos):
    dict_datos = {}
    for nombre, nombre_archivo in archivos.items():
        # verificar que cada archivo existe
        ruta = verificar_archivo(nombre_archivo)
        # cargar datos
        dict_datos[nombre] = pd.read_csv(ruta)

    return dict_datos

""" métodos de análisis de datos """
def mostrar_primeras_filas(nombre: str, datos: pd.DataFrame) -> None:
    # 1.1 mostrar las primeras filas 
    display(Markdown(f"### **Archivo: {nombre}.csv**"))
    display(datos.head())

def mostrar_descripcion(datos: pd.DataFrame) -> None:
    # 1.2
    # cantidad de filas y columnas
    filas, columnas = datos.shape
    display(Markdown(f"**Filas y columnas:** {filas} filas, {columnas} columnas"))

    # tipos de dato
    display(Markdown("#### Tipos de dato por columna"))
    display(datos.dtypes.to_frame(name="tipo_dato").T)

    # cantidad de categorías en las variables categóricas
    categoricas = datos.select_dtypes(include=["object", "category", "string", "bool"])
    if not categoricas.empty:
        display(Markdown("#### Cantidad de categorías (variables categóricas)"))
        display(categoricas.nunique().to_frame(name="cantidad_categorias"))

    # descripción estadística de las continuas
    continuas = datos.select_dtypes(include="number")
    if not continuas.empty:
        display(Markdown("#### Descripción estadística (variables continuas)"))
        display(continuas.describe().T)
    return

def mostrar_duplicados(datos: pd.DataFrame) -> None:
    # 1.3 búsqueda de formularios enviados dos veces
    # y operaciones por duplicado (solo se muestran, no se eliminan)
    # duplicados exactos (todas las columnas iguales)
    exactos = datos.duplicated()
    display(Markdown(
        f"#### Duplicados exactos: {exactos.sum()} filas "
    ))
    if exactos.any():
        display(datos[datos.duplicated(keep=False)].sort_values(list(datos.columns)).head(10))

    # formularios enviados dos veces: mismo id_cliente en la encuesta
    if "id_cliente" in datos.columns and "id_movimiento" not in datos.columns:
        repetidos = datos[datos.duplicated(subset="id_cliente", keep=False)]
        display(Markdown(
            f"#### Formularios con id_cliente repetido: "
            f"{datos.duplicated(subset='id_cliente').sum()} filas de más "
            f"({len(repetidos)} filas involucradas)"
        ))
        if not repetidos.empty:
            display(repetidos.sort_values("id_cliente").head(10))

    # operaciones duplicadas: misma operación con distinto id_movimiento
    if "id_movimiento" in datos.columns:
        display(Markdown(
            f"**id_movimiento repetidos:** {datos.duplicated(subset='id_movimiento').sum()}"
        ))
        clave = [c for c in datos.columns if c != "id_movimiento"]
        operaciones = datos[datos.duplicated(subset=clave, keep=False)]
        display(Markdown(
            f"**Operaciones duplicadas (ignorando id_movimiento):** "
            f"{datos.duplicated(subset=clave).sum()} filas de más "
            f"({len(operaciones)} filas involucradas)"
        ))
        if not operaciones.empty:
            display(operaciones.sort_values(clave).head(10))
    return

def mostrar_faltantes(datos: pd.DataFrame) -> None:
    # 1.4 faltantes por columnas (solo valores nulos reales, isna())
    faltantes = datos.isna().sum()
    resumen = pd.DataFrame({
        "faltantes": faltantes
    })
    resumen = resumen[resumen["faltantes"] > 0].sort_values("faltantes", ascending=False)

    display(Markdown("**Faltantes por columna**"))
    if resumen.empty:
        display(Markdown("No hay valores faltantes."))
        return
    display(resumen)

    # filas afectadas, filtradas por columna con faltantes
    for columna in resumen.index:
        filas_afectadas = datos[datos[columna].isna()]
        display(Markdown(
            f"Filas con `{columna}` faltante: **{len(filas_afectadas)}** "
            f"de {len(datos)} (se muestran las primeras 3)"
        ))
        display(filas_afectadas.head(3))
    return

def mostrar_valores_raros(datos: pd.DataFrame) -> None:
    # 1.5 valores raros 
    # valores raros: no coinciden con el tipo de dato de la columna
    # y formas de decir "no contestó"
    textos = datos.select_dtypes(exclude="number").columns
    fechas = [c for c in textos if c.startswith("fecha")]

    # texto en columnas que deberían ser numéricas
    for columna in textos:
        if columna in fechas:
            continue
        valores = datos[columna].dropna().astype(str)
        es_numero = valores.str.fullmatch(r"\s*\$?\s*-?\d+([.,]\d+)*\s*")
        if valores.empty or es_numero.mean() < 0.5:
            continue  # la columna es realmente de texto
        raros = valores[~es_numero]
        display(Markdown(f"`{columna}` es numérica pero tiene **{len(raros)}** valores que no son números"))
        display(raros.value_counts().to_frame(name="cantidad"))

    # fechas que no son válidas
    for columna in fechas:
        valores = datos[columna].dropna().astype(str)
        raros = valores[pd.to_datetime(valores, errors="coerce", format="mixed").isna()]
        if not raros.empty:
            display(Markdown(f"`{columna}` tiene **{len(raros)}** fechas no válidas"))

    # formas de decir "no contestó": isna() solo detecta una
    marcadores = ["prefiero no contestar", "s/d", "n/c", "ns/nc", "no sabe", "no contesta", "-", "?", ""]
    centinelas = [-999, -99, 999, 9999]
    resumen = []
    for columna in datos.columns:
        if columna.startswith("id_"):
            continue
        if columna in textos:
            disfrazados = datos[columna].astype(str).str.strip().str.lower().isin(marcadores)
        else:
            disfrazados = datos[columna].isin(centinelas)
        if disfrazados.any():
            resumen.append((columna, datos[columna].isna().sum(), disfrazados.sum()))

    display(Markdown("**Formas de decir «no contestó»: isna() frente al total real**"))
    if not resumen:
        display(Markdown("No se encontraron marcadores de no respuesta."))
        return
    tabla = pd.DataFrame(
        [(c, n, d) for c, n, d in resumen],
        columns=["columna", "nulos_isna", "disfrazados"],
    ).set_index("columna")
    tabla["total_no_respuesta"] = tabla.sum(axis=1)
    display(tabla)
    return

def mostrar_rangos_imposibles(datos: pd.DataFrame) -> None:
    # 1.5 valores raros y rangos imposibles
    # numéricas fuera de su rango posible (solo se detectan, no se corrigen)
    infinito = float("inf")
    rangos = {
        "edad": (18, 100),
        "patrimonio_invertible_usd": (0, infinito),
        "meses_gastos_cubiertos": (0, infinito),
        "anios_experiencia_invirtiendo": (0, infinito),
        "puntaje_test_conocimiento": (0, 10),
        "conocimiento_autopercibido": (1, 5),
        "monto": (0, infinito),
    }

    display(Markdown("**Numéricas con rangos imposibles**"))
    encontrados = 0
    for columna, (minimo, maximo) in rangos.items():
        if columna not in datos.columns:
            continue
        fuera = datos[columna].notna() & ~datos[columna].between(minimo, maximo)
        if fuera.any():
            encontrados += 1
            display(Markdown(
                f"`{columna}` fuera de [{minimo}, {maximo}]: **{fuera.sum()}** valores "
                f"(desde {datos.loc[fuera, columna].min()} hasta {datos.loc[fuera, columna].max()})"
            ))

    # no se puede invertir más años que los vividos
    if "edad" in datos.columns and "anios_experiencia_invirtiendo" in datos.columns:
        incoherentes = datos["anios_experiencia_invirtiendo"] > datos["edad"]
        if incoherentes.any():
            encontrados += 1
            display(Markdown(f"`anios_experiencia_invirtiendo` mayor que `edad`: **{incoherentes.sum()}** filas"))

    if encontrados == 0:
        display(Markdown("No se encontraron rangos imposibles."))
    return

def mostrar_campos_vacios(datos: pd.DataFrame) -> None:
    # 1.6 campos vacíos
    return

def validar_faltantes(datos: pd.DataFrame) -> None:
    # 1.7 faltantes de ingresos y patrimonios repartidos
    return

def validar_patrimonio(datos: pd.DataFrame) -> None:
    # 1.7 faltantes de ingresos y patrimonios repartidos
    return

def mostrar_datos(dict_datos) -> None:
    # imprimir tipos de datos
    for nombre, datos in dict_datos.items():

        # 1.1 mostrar las primeras filas 
        mostrar_primeras_filas(nombre, datos)

        # 1.2 cantidad de filas y columnas, tipos de datos, 
        # cantidad de categorías y descripción de las continuas
        mostrar_descripcion(datos)

        # 1.3 búsqueda de formularios enviados dos veces
        # y operaciones por duplicado
        mostrar_duplicados(datos)

        # 1.4 faltantes por columnas
        mostrar_faltantes(datos)

        # 1.5 valores raros y rangos imposibles
        mostrar_valores_raros(datos)
        mostrar_rangos_imposibles(datos)

        # 1.6 campos vacíos
        mostrar_campos_vacios(datos)

        # 1.7 faltantes de ingresos y patrimonios repartidos
        validar_faltantes(datos)
        validar_patrimonio(datos) 

    return

""" métodos generales """
def main() -> None:
    # nombre de los archivos
    archivos = {
        "encuesta_inversores": "encuesta_inversores.csv",
        "movimientos_cartera": "movimientos_cartera.csv",
        "catalogo_instrumentos": "catalogo_instrumentos.csv",
        "tasas_referencia": "tasas_referencia.csv",
    }
    # cargar datos
    dict_datos = cargar_datos(archivos)

    # mostrar resultados
    mostrar_datos(dict_datos)

if __name__ == "__main__":
    main()


### **Archivo: encuesta_inversores.csv**

,id_cliente,fecha_respuesta,edad,sexo,departamento,cantidad_dependientes,nivel_educativo,ocupacion,ingreso_mensual_liquido,patrimonio_invertible_usd,...,reaccion_caida_20,objetivo_principal,fuente_informacion,prioridad_plazo_fijo,prioridad_letras_bcu,prioridad_fondos_locales,prioridad_obligaciones,prioridad_etf_exterior,prioridad_inmuebles,prioridad_criptomonedas
0,20504,2026-06-24,52.0,Masculino,Montevideo,0,Terciaria completa,Profesional independiente,141700,4200.0,...,Compraría más,Crecimiento del capital,Amigos y familia,7,6,4,5,1,3,2
1,20466,2026-03-29,62.0,Masculino,Colonia,0,Terciaria completa,Jubilado,$ 131.300,13900.0,...,No haría nada,Generar una renta mensual,Investigación propia,7,6,4,5,2,3,1
2,20432,2026-01-11,40.0,Masculino,Maldonado,0,Terciaria completa,Jubilado,53700,4600.0,...,No haría nada,Complemento jubilatorio,Asesor,7,6,4,3,2,5,1
3,20052,2026-02-21,48.0,Masculino,Canelones,0,Posgrado,Empresario / dueño de negocio,0,27700.0,...,No haría nada,Compra de vivienda,Asesor financiero,7,4,6,5,2,3,1
4,20084,2026-04-01,82.0,Femenino,Canelones,0,Posgrado,Jubilado,85100,12100.0,...,No haría nada,Complemento jubilatorio,Amigos y familia,7,6,3,4,1,5,2


**Filas y columnas:** 657 filas, 31 columnas

#### Tipos de dato por columna

,id_cliente,fecha_respuesta,edad,sexo,departamento,cantidad_dependientes,nivel_educativo,ocupacion,ingreso_mensual_liquido,patrimonio_invertible_usd,...,reaccion_caida_20,objetivo_principal,fuente_informacion,prioridad_plazo_fijo,prioridad_letras_bcu,prioridad_fondos_locales,prioridad_obligaciones,prioridad_etf_exterior,prioridad_inmuebles,prioridad_criptomonedas
tipo_dato,int64,str,float64,str,str,int64,str,str,str,float64,...,str,str,str,int64,int64,int64,int64,int64,int64,int64


...

#### Descripción estadística (variables continuas)

,count,mean,std,min,25%,50%,75%,max
valor,8.0,10.5,13.242006,2.7,4.4,5.975,8.8625,42.8


#### Duplicados exactos: 0 filas 

### Faltantes por columna

No hay valores faltantes.